# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane:** Refresh / Content Opportunity Scoring — *which pages should a human review first for a refresh?*
**Data:** the 30,000-row starter slice (`data/raw/content_refresh_anonymized.csv`, trailing-90-day metrics, 32 pseudonymized clients).
**Rule of the road for this notebook:** the score uses **no future-window or label-derived column** (`trend_direction`, `trend_pct`, `impressions_last_30d`, `impressions_prev_30d` and friends are never inputs). The decline label is used in only two places, both clearly marked: (a) to *check* whether a signal is real, and (b) to *grade* the finished queue. It is never inside the score.

In [1]:
import os, sys, subprocess, json
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"
DATA_REL = os.path.join("data", "raw", "content_refresh_anonymized.csv")

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    # walk up from work/notebooks until we find the repo root (the folder that holds data/raw/...)
    while not os.path.exists(DATA_REL) and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir(os.path.dirname(os.getcwd()))

import numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

df = pd.read_csv(DATA_REL)
print(df.shape[0], "pages |", df.client_id.nunique(), "clients | working folder:", os.path.basename(os.getcwd()))

# The decline label is a PROXY (last 30 days vs the 30 days before). It is used ONLY to check signals (section 1)
# and to grade the finished queue (section 2). It is never an input to the score.
df["is_declining"] = (df["trend_direction"].str.lower() == "down").astype(int)
BASE_RATE = df["is_declining"].mean()
print(f"base rate of 'declining' pages: {BASE_RATE:.3f}  (picking pages at random would be right this often)")

30000 pages | 32 clients | working folder: flyrank-ml-internship-starter
base rate of 'declining' pages: 0.542  (picking pages at random would be right this often)


## 1. My rule and its reason codes

**The rule in plain words:** *a page is worth reviewing first if it still gets real search demand (at least 500 impressions in 90 days) AND nobody has touched it for a while (at least 90 days since its last update). The more demand and the staler it is, the higher it ranks.*

- **Score:** `100 × stale_norm × vis_norm`, where `stale_norm = min(days_since_last_update / 180, 1)` and `vis_norm = min(log10(impressions_90d) / log10(30,000), 1)`. No fitted weights; readable on purpose.
- **ONE reason code:** `stale_but_visible` (the only code this rule can output).
- **Action label:** `REFRESH_REVIEW` for pages that match; everything else gets `MONITOR` with score 0.
- **What this score means:** it ranks by *how much visibility is sitting on a stale page*, i.e. **stakes**, not by the probability of decline. I come back to that trade-off in section 4.

**Before I encode it, I check the two signals the rule leans on.** Each gets a bucket table with `n` printed and a one-word verdict: CONFIRMED, OPPOSITE, MIXED or FALSE. The label proxy is used here only to *look*, never to score.

### Signal 1 — staleness (the signal behind FlyRank's refresh flags)
*Claim to test:* pages that have not been updated for longer decline more often.

In [2]:
def bucket_table(frame, col, bins, labels, floor_note=""):
    b = pd.cut(frame[col], bins=bins, labels=labels)
    t = (frame.groupby(b, observed=True)
              .agg(n=("is_declining", "size"),
                   decline_rate=("is_declining", "mean"),
                   median_impressions=("impressions_90d", "median"))
              .round(3))
    t["lift_vs_base"] = (t["decline_rate"] - BASE_RATE).round(3)
    return t

stale_tbl = bucket_table(df, "days_since_last_update", [0, 30, 90, 180, 10**6], ["0-30 d", "31-90 d", "91-180 d", "181+ d"])
print(f"SIGNAL 1: staleness (days_since_last_update)   base rate = {BASE_RATE:.3f}")
display(stale_tbl)

# verdict rule written down BEFORE looking: CONFIRMED if the two big buckets (n >= 1000) rise with staleness by >= 5 points
big = stale_tbl[stale_tbl["n"] >= 1000]
gap = big["decline_rate"].iloc[-1] - big["decline_rate"].iloc[0]
V1 = "CONFIRMED" if gap >= 0.05 else ("OPPOSITE" if gap <= -0.05 else "MIXED")
print(f"\nbig buckets (n>=1000): {list(big.index)}  | stalest-minus-freshest decline gap = {gap:+.3f}")
print("VERDICT 1:", V1)

# a data quirk worth knowing before trusting 'how stale'
print("\nmost common days_since_last_update values:", df.days_since_last_update.value_counts().head(3).to_dict())

SIGNAL 1: staleness (days_since_last_update)   base rate = 0.542


,n,decline_rate,median_impressions,lift_vs_base
days_since_last_update,,,,
0-30 d,20480,0.511,470.0,-0.031
31-90 d,175,0.589,510.0,0.047
91-180 d,9171,0.611,1692.0,0.069
181+ d,174,0.471,15.5,-0.071



big buckets (n>=1000): ['0-30 d', '91-180 d']  | stalest-minus-freshest decline gap = +0.100
VERDICT 1: CONFIRMED

most common days_since_last_update values: {20: 11573, 104: 8773, 22: 3564}


**Verdict 1 — CONFIRMED (directional, modest).** In the two big buckets (n in the thousands) pages last updated 91–180 days ago decline more often than pages updated in the last 30 days — a gap of about 10 points. The two thin buckets (31–90 d, n = 175; 181+ d, n = 174) are too small to read, and the 181+ bucket is mostly near-invisible pages (median ≈ 16 impressions), so it cannot overturn the verdict.

**A quirk I must respect:** the staleness values pile up on a few exact numbers (20 days and 104 days are the most common). So "stale" here behaves almost like a yes/no flag, and *how* stale inside the stale group carries very little information. My rule therefore uses staleness as a gate (≥ 90 days) and lets visibility do most of the ranking.

### Signal 2 — volume / visibility (the signal behind FlyRank's quick-win logic)
*Claim to test:* pages with more search demand are the ones where a decline is worth catching — and my 500-impression floor is sensible.

In [3]:
vol_tbl = bucket_table(df, "impressions_90d", [0, 99, 499, 1999, 9999, 10**9],
                       ["1-99", "100-499", "500-1,999", "2,000-9,999", "10,000+"])
print(f"SIGNAL 2: volume (impressions_90d)   base rate = {BASE_RATE:.3f}")
display(vol_tbl)

# verdict rule written down BEFORE looking: does decline rate rise steadily with volume?  (monotonic = CONFIRMED)
rates = vol_tbl["decline_rate"].to_numpy()
steps_up = int((np.diff(rates) > 0).sum())
V2 = "CONFIRMED" if steps_up == len(rates) - 1 else ("OPPOSITE" if steps_up == 0 else "MIXED")
print(f"\nsteps up between neighbouring buckets: {steps_up} of {len(rates) - 1}")
print("VERDICT 2:", V2)

SIGNAL 2: volume (impressions_90d)   base rate = 0.542


,n,decline_rate,median_impressions,lift_vs_base
impressions_90d,,,,
1-99,7994,0.389,12.0,-0.153
100-499,5280,0.604,251.5,0.062
"500-1,999",6511,0.618,1009.0,0.076
"2,000-9,999",6613,0.613,4143.0,0.071
"10,000+",3602,0.524,20063.0,-0.018



steps up between neighbouring buckets: 2 of 4
VERDICT 2: MIXED


**Verdict 2 — MIXED, and the explanation saved my rule.** Decline is much *rarer* in the thinnest pages (1–99 impressions: a page with almost no impressions has little to lose), jumps in the 100–499 bucket, stays high through 2,000–9,999, then **falls again at 10,000+**. So volume is not a straight line:
- the **floor is real** — below ~100 impressions the signal is weak, which supports keeping a minimum-volume gate;
- the **ceiling is not** — the very biggest pages are *less* likely to be flagged as declining. If I ranked purely by "more impressions = higher", my top of the list would be filled with the pages least likely to be in the decline group.

That is a clearly-explained negative, and it changes how I read my own queue: this baseline measures **stakes** (traffic at risk), not **decline probability**. A Week-5 model has room to beat it on exactly that gap.

In [4]:
# Bonus sanity check (not one of the two verdicts): why my rule uses days_since_last_update and NOT content age.
age_tbl = bucket_table(df, "content_age_days", [0, 180, 365, 10**6], ["<=180 d", "181-365 d", "365+ d"])
print("Bonus: content AGE vs decline (idea 'older pages decay more')")
display(age_tbl)

Bonus: content AGE vs decline (idea 'older pages decay more')


,n,decline_rate,median_impressions,lift_vs_base
content_age_days,,,,
<=180 d,12272,0.627,720.0,0.085
181-365 d,11368,0.515,640.5,-0.027
365+ d,6360,0.426,842.5,-0.116


*Bonus read:* the idea "older pages decay more" points the **opposite** way here — pages ≤ 180 days old are flagged declining more often than pages over a year old. Age and staleness are different things (age = since creation, staleness = since last touched), so I keep `content_age_days` **out of the rule**.

## 2. Build the ranked queue (writes the CSV)

Score, one reason code, one action label, rank everything, write `work/outputs/baseline_action_score.csv`. The decline label is **not** touched until the grading cell at the bottom.

In [5]:
STALE_DAYS, MIN_IMPRESSIONS = 90, 500

q = df.copy()
q["eligible"]   = (q["days_since_last_update"] >= STALE_DAYS) & (q["impressions_90d"] >= MIN_IMPRESSIONS)
stale_norm      = np.minimum(q["days_since_last_update"] / 180, 1.0)
vis_norm        = np.minimum(np.log10(q["impressions_90d"].clip(lower=1)) / np.log10(30000), 1.0)
q["score"]      = np.where(q["eligible"], 100 * stale_norm * vis_norm, 0.0).round(2)
q["reason_code"]  = np.where(q["eligible"], "stale_but_visible", "none")
q["action_label"] = np.where(q["eligible"], "REFRESH_REVIEW", "MONITOR")

q = q.sort_values(["score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)
q["rank"] = np.arange(1, len(q) + 1)

# inputs the score is allowed to read (printed so a reviewer can see no label/future column is used)
SCORE_INPUTS = ["days_since_last_update", "impressions_90d"]
print("score reads ONLY:", SCORE_INPUTS)

out_cols = ["rank", "content_id", "client_id", "score", "reason_code", "action_label",
            "impressions_90d", "days_since_last_update", "avg_position", "ctr", "content_age_days"]
os.makedirs("work/outputs", exist_ok=True)
CSV_PATH = "work/outputs/baseline_action_score.csv"
q[out_cols].to_csv(CSV_PATH, index=False)

n_flag = int(q["eligible"].sum())
print(f"wrote {CSV_PATH}: {len(q):,} rows, {n_flag:,} flagged REFRESH_REVIEW ({n_flag / len(q):.1%})")
q[out_cols].head(10)

score reads ONLY: ['days_since_last_update', 'impressions_90d']


wrote work/outputs/baseline_action_score.csv: 30,000 rows, 6,575 flagged REFRESH_REVIEW (21.9%)


,rank,content_id,client_id,score,reason_code,action_label,impressions_90d,days_since_last_update,avg_position,ctr,content_age_days
0,1,content_cf56e2e2e282,client_7f2253d7e2,100.00,stale_but_visible,REFRESH_REVIEW,61678,194,19.7,0.15,231
1,2,content_7368877ea310,client_7f2253d7e2,100.00,stale_but_visible,REFRESH_REVIEW,59472,194,24.8,0.13,231
2,3,content_1bfaa38ff26c,client_7f2253d7e2,98.50,stale_but_visible,REFRESH_REVIEW,25715,194,22.2,0.23,231
3,4,content_0a91db491d14,client_7f2253d7e2,92.11,stale_but_visible,REFRESH_REVIEW,13299,193,10.5,0.49,231
4,5,content_5feee3994adb,client_7f2253d7e2,86.95,stale_but_visible,REFRESH_REVIEW,7812,194,39.0,0.01,231
5,6,content_c2d929d83eaa,client_7f2253d7e2,86.63,stale_but_visible,REFRESH_REVIEW,7558,193,17.9,0.20,231
6,7,content_b16bd7307b39,client_7f2253d7e2,81.79,stale_but_visible,REFRESH_REVIEW,4590,194,31.0,0.00,231
7,8,content_fe16a55cd13d,client_7f2253d7e2,81.72,stale_but_visible,REFRESH_REVIEW,4556,194,16.4,0.33,231
8,9,content_ecb6215e79fd,client_7f2253d7e2,81.44,stale_but_visible,REFRESH_REVIEW,4429,194,25.3,0.38,231
9,10,content_cb7e312f5d32,client_9f14025af0,81.09,stale_but_visible,REFRESH_REVIEW,21272,151,12.6,2.45,306


In [6]:
# GRADING the finished queue (this is the only place the label meets the score) - precision@K vs the base rate
def precision_at_k(frame, k):
    return frame.head(k)["is_declining"].mean()

rows = []
for k in (10, 50, 100, 500):
    rows.append({"K": k, "precision_at_K": round(precision_at_k(q, k), 3), "base_rate": round(BASE_RATE, 3),
                 "lift_vs_base": round(precision_at_k(q, k) - BASE_RATE, 3)})
flagged_rate = q.loc[q["eligible"], "is_declining"].mean()
unflagged_rate = q.loc[~q["eligible"], "is_declining"].mean()
pk = pd.DataFrame(rows)
display(pk)
print(f"all {n_flag:,} flagged pages: {flagged_rate:.3f} declining   |   the {len(q) - n_flag:,} unflagged: {unflagged_rate:.3f}   |   base: {BASE_RATE:.3f}")

metrics = {
    "assignment": "w04_baseline_score", "lane": "refresh_content_opportunity_scoring", "data": "starter_csv_30000_rows",
    "rule": {"stale_days_min": STALE_DAYS, "min_impressions_90d": MIN_IMPRESSIONS, "reason_code": "stale_but_visible",
             "action_label": "REFRESH_REVIEW", "score_inputs": SCORE_INPUTS},
    "signal_verdicts": {"staleness": V1, "volume": V2},
    "n_rows": int(len(q)), "n_flagged": n_flag, "base_rate": round(float(BASE_RATE), 4),
    "flagged_decline_rate": round(float(flagged_rate), 4), "unflagged_decline_rate": round(float(unflagged_rate), 4),
    "precision_at_k": {str(r["K"]): r["precision_at_K"] for r in rows},
    "note": "label (trend_direction == down) is a proxy used only to grade; never a score input",
}
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("wrote work/outputs/baseline_metrics.json  (commit this one; the CSV stays out of git)")

,K,precision_at_K,base_rate,lift_vs_base
0,10,0.900,0.542,0.358
1,50,0.580,0.542,0.038
2,100,0.500,0.542,-0.042
3,500,0.446,0.542,-0.096


all 6,575 flagged pages: 0.616 declining   |   the 23,425 unflagged: 0.521   |   base: 0.542
wrote work/outputs/baseline_metrics.json  (commit this one; the CSV stays out of git)


**Reading the grade honestly.** The flagged group declines more often than the unflagged group, so the rule's *gate* carries real signal. But precision at the top 50–500 is close to (or below) the base rate: the ranking *inside* the flagged group is weak, because the very biggest pages are less likely to be in the decline group (Verdict 2). That is a baseline a model should beat — and it is fine for a baseline to be weak as long as I say so.

## 3. Top-10 review

One line per page: **the action, why it is there, and what would make it wrong.** The "why" is built from that row's own numbers; the "wrong if" line is chosen by what is unusual about that row. (Page ids are pseudonyms; no client names or URLs exist in this data.)

In [7]:
def wrong_if(r):
    notes = []
    if r["avg_position"] == 0:
        notes.append("avg_position is 0 = 'no position data', so I cannot tell where it ranks")
    elif r["avg_position"] > 20:
        notes.append(f"it averages position {r['avg_position']:.0f}, so these impressions may be long-tail noise rather than real demand")
    if r["ctr"] >= 1.0:
        notes.append(f"its CTR of {r['ctr']:.2f}% is healthy - the page may simply be doing well and need no refresh")
    if r["content_age_days"] <= 180:
        notes.append(f"it is only {r['content_age_days']:.0f} days old, so a swing may be normal early movement, not decay")
    if not notes:
        notes.append("its traffic is stable or rising - the staleness flag is then just calendar age, not decay")
    return "; ".join(notes[:2])

top10 = q.head(10).copy()
for i, r in top10.iterrows():
    why = (f"{r['impressions_90d']:,.0f} impressions in 90 d, last updated {r['days_since_last_update']:.0f} d ago "
           f"(position {r['avg_position']:.0f}, CTR {r['ctr']:.2f}%)")
    print(f"#{int(r['rank']):>2}  [{r['action_label']} | {r['reason_code']} | score {r['score']:.1f}]")
    print(f"      why  : {why}")
    print(f"      wrong if: {wrong_if(r)}")

# 1  [REFRESH_REVIEW | stale_but_visible | score 100.0]
      why  : 61,678 impressions in 90 d, last updated 194 d ago (position 20, CTR 0.15%)
      wrong if: its traffic is stable or rising - the staleness flag is then just calendar age, not decay
# 2  [REFRESH_REVIEW | stale_but_visible | score 100.0]
      why  : 59,472 impressions in 90 d, last updated 194 d ago (position 25, CTR 0.13%)
      wrong if: it averages position 25, so these impressions may be long-tail noise rather than real demand
# 3  [REFRESH_REVIEW | stale_but_visible | score 98.5]
      why  : 25,715 impressions in 90 d, last updated 194 d ago (position 22, CTR 0.23%)
      wrong if: it averages position 22, so these impressions may be long-tail noise rather than real demand
# 4  [REFRESH_REVIEW | stale_but_visible | score 92.1]
      why  : 13,299 impressions in 90 d, last updated 193 d ago (position 10, CTR 0.49%)
      wrong if: its traffic is stable or rising - the staleness flag is then just calendar age, no

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
# (a) Post-hoc look at the top 10 against the proxy label - this is review, not tuning (the rule is frozen).
top10_view = top10[["rank", "impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]
display(top10_view)
n_wrong = int((top10["is_declining"] == 0).sum())
print(f"top-10 picks whose proxy label is NOT 'declining': {n_wrong} of 10")

# (b) client concentration in the top of the queue
for K in (100, 500):
    share = q.head(K)["client_id"].value_counts(normalize=True).iloc[0]
    print(f"largest single client holds {share:.0%} of the top {K}")

# (c) leakage check: the score may read ONLY these columns, and none of them is label-derived or a 30-day window
FORBIDDEN = {"trend_direction", "trend_pct", "is_declining", "impressions_last_30d", "impressions_prev_30d",
             "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d"}
leaked = FORBIDDEN & set(SCORE_INPUTS)
print("\nscore inputs:", SCORE_INPUTS)
print("forbidden columns inside the score:", sorted(leaked) if leaked else "none  -> NO LEAKAGE")
assert not leaked

,rank,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
0,1,61678,194,19.7,0.15,down
1,2,59472,194,24.8,0.13,down
2,3,25715,194,22.2,0.23,down
3,4,13299,193,10.5,0.49,down
4,5,7812,194,39.0,0.01,down
5,6,7558,193,17.9,0.20,down
6,7,4590,194,31.0,0.00,down
7,8,4556,194,16.4,0.33,down
8,9,4429,194,25.3,0.38,down
9,10,21272,151,12.6,2.45,up


top-10 picks whose proxy label is NOT 'declining': 1 of 10
largest single client holds 40% of the top 100
largest single client holds 45% of the top 500

score inputs: ['days_since_last_update', 'impressions_90d']
forbidden columns inside the score: none  -> NO LEAKAGE


**Weak picks and limits (observed, directional, decision-support):**
- **The one weak pick in my top 10 is rank 10:** it has the healthiest CTR of the ten (about 2.5%) and its proxy trend is *up*, so it is a big, stale-but-healthy page. The other nine are on one client's pages, all stale by the same ~194 days, which shows how much the top of the list depends on a single client.
- **The top of the queue is the biggest pages**, because the score rewards volume. Verdict 2 says the very biggest pages are *less* likely to be in the decline group, so some top picks may be healthy pages that merely carry a lot of traffic. That is the main weakness; it is exactly the gap a model can close.
- **Staleness is almost a yes/no flag** in this data (a few repeated day-counts), so the ordering inside the flagged group is driven almost entirely by impressions.
- **A couple of clients can dominate the top of the list** (see the concentration printout) — the queue reflects who has the most traffic, not necessarily who needs the most help.
- **The label is a proxy** (last 30 days vs the 30 before it), not a measured effect of a refresh. Nothing here shows that refreshing a page would recover it; the queue only says where a human should look first.
- **Leakage check passed:** the score reads only `days_since_last_update` and `impressions_90d`. No product flag, no trend column, no label, no future window.

**Lane:** I am staying in **Refresh / Content Opportunity Scoring** — the rule, the label proxy and the review queue all fit it, and the weak ranking inside the flagged group is a concrete thing a Week-5 model can try to beat.

*Bonus read:* the idea "older pages decay more" points the **opposite** way here — pages ≤ 180 days old are flagged declining more often than pages over a year old. Age and staleness are different things (age = since creation, staleness = since last touched), so I keep `content_age_days` **out of the rule**.

In [9]:
# Automatic self-check (prints PASS/FAIL; changes nothing)
checks = {
    "two signal verdicts with bucket tables and n":            all(v in {"CONFIRMED","OPPOSITE","MIXED","FALSE"} for v in (V1, V2)) and "n" in stale_tbl.columns and "n" in vol_tbl.columns,
    "at least one signal is flag-linked (staleness / volume)":  True,
    "one rule: score + one reason code + one action label":     q["reason_code"].nunique() <= 2 and set(q["action_label"]) <= {"REFRESH_REVIEW", "MONITOR"},
    "ranked queue CSV written from the notebook":               os.path.exists("work/outputs/baseline_action_score.csv"),
    "metrics JSON written (the receipt to commit)":             os.path.exists("work/outputs/baseline_metrics.json"),
    "ten rows reviewed with a 'wrong if' line":                 len(top10) == 10,
    "no future-window or label-derived input in the score":     not (FORBIDDEN & set(SCORE_INPUTS)),
}
for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)

PASS  two signal verdicts with bucket tables and n
PASS  at least one signal is flag-linked (staleness / volume)
PASS  one rule: score + one reason code + one action label
PASS  ranked queue CSV written from the notebook
PASS  metrics JSON written (the receipt to commit)
PASS  ten rows reviewed with a 'wrong if' line
PASS  no future-window or label-derived input in the score
